# Chaos Calculator — motor en notebook

Este notebook es para usar el motor sin la interfaz CRT. La idea es simple: elegir un sistema, modificar parámetros, generar una condición inicial razonable, integrar y graficar sin navegar menús.

In [ ]:
import os
os.environ["CHAOS_AUDIO_AUTOSTART"] = "0"  # el notebook no lanza música

import numpy as np
import matplotlib.pyplot as plt

from engine import chaos_runtime as cc

cc.apply_style()
print("Presets disponibles:")
for key in sorted(cc.PRESETS.keys(), key=lambda x: int(x)):
    p = cc.PRESETS[key]
    print(f"{key}. {p['name']} :: {p.get('desc','')}")

## 1. Elegir sistema

Cambia `PRESET_ID`. Algunos buenos para probar rápido:

- `5`: Lorenz63.
- `6`: Rössler.
- `7`: Duffing forzado.
- `4`: tres cuerpos figura-8.
- `3`: Hénon-Heiles.

In [ ]:
PRESET_ID = "5"  # cambia esto
cfg = cc.PRESETS[PRESET_ID]
sysm = cc.build_system(cfg)

print(sysm.name)
print(sysm.desc)
print("kind:", getattr(sysm, "kind", "?"))
print("dim:", getattr(sysm, "dim", len(getattr(sysm, "state_labels", []))))
print("variables:", getattr(sysm, "state_labels", []))

## 2. Parámetros de integración

Puedes subir `T` o bajar `dt_out`, pero eso aumenta el costo. Para exploración inicial, no conviene partir con valores gigantes.

In [ ]:
T = float(getattr(sysm, "t_final", None) or cfg.get("t_final", cc.CONFIG.get("ode_T", 80.0)))
dt_out = 0.03

# Para probar rápido:
# T = 20
# dt_out = 0.05

print("T =", T)
print("dt_out =", dt_out)

## 3. Condición inicial inteligente

El motor intenta elegir una condición inicial aleatoria que no sea absurda para el sistema. Puedes fijar semilla para reproducibilidad.

In [ ]:
z0, msg = cc.intelligent_random_initial_condition(
    sysm,
    attempts=250,
    seed=1234,
    sanity=True,
)
print(msg)
print(z0)
print(cc.validate_initial_condition(sysm, z0))

## 4. Integrar trayectoria

In [ ]:
traj = cc.Integrator(sysm).integrate(z0, T, dt_out, method="DOP853")
t = traj.t
Z = traj.z
print("samples:", Z.shape)
print("t final:", t[-1])

## 5. Series temporales

In [ ]:
labels = getattr(sysm, "state_labels", [f"z{i}" for i in range(Z.shape[1])])
plt.figure(figsize=(10, 4))
for i in range(min(4, Z.shape[1])):
    plt.plot(t, Z[:, i], label=labels[i])
plt.xlabel("t")
plt.ylabel("estado")
plt.title(f"{sysm.name}: series temporales")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Proyección de fase 2D/3D

In [ ]:
# Proyección 2D
ix, iy = 0, 1
plt.figure(figsize=(6, 5))
plt.plot(Z[:, ix], Z[:, iy], lw=0.8)
plt.xlabel(labels[ix])
plt.ylabel(labels[iy])
plt.title(f"{sysm.name}: proyección de fase")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Proyección 3D si la dimensión alcanza
if Z.shape[1] >= 3:
    from mpl_toolkits.mplot3d import Axes3D  # noqa
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(Z[:, 0], Z[:, 1], Z[:, 2], lw=0.7)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_zlabel(labels[2])
    ax.set_title(f"{sysm.name}: trayectoria 3D")
    plt.show()

## 7. Energía o drift, si existe

In [ ]:
try:
    E = sysm.energy(Z)
    if np.all(np.isfinite(E)):
        drift = E - E[0]
        plt.figure(figsize=(8, 4))
        plt.plot(t, drift)
        plt.xlabel("t")
        plt.ylabel("E(t)-E(0)")
        plt.title(f"{sysm.name}: drift energético")
        plt.grid(True, alpha=0.3)
        plt.show()
        print("E0 =", E[0], "Ef =", E[-1], "max |drift| =", np.max(np.abs(drift)))
    else:
        print("El sistema no tiene energía finita definida para esta trayectoria.")
except Exception as exc:
    print("Sin energía definida:", exc)

## 8. FTLE y SALI

Esto puede tardar más que integrar una sola trayectoria porque integra ecuaciones variacionales.

In [ ]:
T_lyap = min(60.0, T)
ly = cc.LyapunovAnalyzer(sysm)
tl, ftle, sali = ly.run(z0, T_lyap, dt_out=0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(tl, ftle)
ax[0].set_title("FTLE")
ax[0].set_xlabel("t")
ax[0].set_ylabel(r"$\lambda_1(t)$")
ax[0].grid(True, alpha=0.3)

ax[1].semilogy(tl, sali)
ax[1].set_title("SALI")
ax[1].set_xlabel("t")
ax[1].set_ylabel("SALI")
ax[1].grid(True, alpha=0.3)
plt.show()

## 9. Exportar trayectoria y metadata

In [ ]:
import csv, json
from pathlib import Path

out = Path("data/notebook_exports") / sysm.name
out.mkdir(parents=True, exist_ok=True)

with open(out / "trajectory.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["t"] + labels)
    for ti, zi in zip(t, Z):
        writer.writerow([ti] + list(zi))

meta = {
    "system": sysm.name,
    "description": getattr(sysm, "desc", ""),
    "T": T,
    "dt_out": dt_out,
    "z0": z0.tolist(),
}
(out / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("exportado en", out)

## 10. Definir un sistema manual rápido

Ejemplo: Lorenz escrito a mano. Puedes reemplazar las ecuaciones por cualquier ODE autónoma `z'=f(z)`.

In [ ]:
# variables = "x y z"
# equations = ["sigma*(y-x)", "x*(rho-z)-y", "x*y-beta*z"]
# params = {"sigma": 10, "rho": 28, "beta": 8/3}
# manual_cfg = dict(
#     kind="ode",
#     name="Mi_Lorenz",
#     desc="Lorenz definido manualmente",
#     variables=variables,
#     equations=equations,
#     params=params,
#     z0=[1, 1, 1],
#     t_final=40,
#     ic_bounds=[[-20,20],[-30,30],[0,55]],
# )
# sys_manual = cc.build_system(manual_cfg)
# z0_manual, msg = cc.intelligent_random_initial_condition(sys_manual, seed=1)
# traj_manual = cc.Integrator(sys_manual).integrate(z0_manual, 40, 0.03, "DOP853")
# print(msg, traj_manual.z.shape)